## What is Rag Evalution?

In this context we are going to create a method to evaluate a RAG system for accuracy. 

We are going to evaluate this by generating an artifical dataset using and LLM and then we are going to use a technique known as LLM as a judge to measure the accuracy.

## LLM as a judge?

This technique from my understanding right now we use an input and compare an output from an LLM to a expected output and we are going to ask the LLM to generate an accuracy amount and we will then describe this ammount through the pandas library.

# Install Dependencies


In [ ]:
# Test if pip works at all
%pip --version

In [ ]:
# Then the rest
%pip install --verbose torch transformers langchain sentence-transformers tqdm openpyxl openai pandas datasets langchain-community ragatouille

In [ ]:
# Install torch first (this is the big one)
%pip install torch --progress-bar on

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from tqdm.auto import tqdm

import pandas as pd
from typing import Optional, List, Tuple
import json
import datasets

pd.set_option("display.max_colwidth", None)

The panda method that is listed above is an method that is used to set the display view of how the panda library will display the data.

its a 2 parameter method.
pd.set_option('display.max_colwidth', 500) 

This method above will essentially set the max col width to about 500 characters, but you have the option to leave this parameter to 'None' and this will essentially not limit the use of the display colulmn.


In [ ]:
%pip install ipywidgets


## Hugging face Hub
we are going to use the hugging face hub to load machine-learning asssets, models, datasets, and demos. 

In this specific example we are going to login to the hugging face hub and we are going to use a public dataset.

In [15]:
from huggingface_hub import notebook_login

notebook_login()

When loading the dataset through huggingface we can set the split to "train" to use just for training data.

In [ ]:
ds = datasets.load_dataset("m-ric/huggingface_doc", split="train")

Now that we have loaded a dataset we can build questions and associated context, we can ask an LLM to generate questions based on this context.

In [ ]:
%pip install langchain

In [ ]:
%pip install langchain-text-splitters langchain-core

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document as LangchainDocument

langchain_docs = [
    LangchainDocument(page_content=doc["text"], metadata={"source": doc["source"]})
    for doc in tqdm(ds)
]


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    add_start_index=True,
    separators=["\n\n", "\n", ".", " ", ""],
)

docs_processed = []
for doc in langchain_docs:
    docs_processed += text_splitter.split_documents([doc])

## Using agents for Questions Generation.

We are going to use an LLM agent in order to generate questions based on the training data that we loaded in. 

In [17]:
from huggingface_hub import InferenceClient

# Use a supported model instead
repo_id = "meta-llama/Llama-3.2-3B-Instruct"  # or another active model
llm_client = InferenceClient(model=repo_id, timeout=120)

def call_llm(inference_client: InferenceClient, prompt: str):
    messages = [{"role": "user", "content": prompt}]
    response = inference_client.chat_completion(
        messages=messages,
        max_tokens=1000,
    )
    return response.choices[0].message.content

# Test it
call_llm(llm_client, "This is a test context")

"It looks like you're testing the system. How can I assist you today?"

In [18]:
QA_generation_prompt = """
Your task is to write a factoid question and an answer given a context.
Your factoid question should be answerable with a specific, concise piece of factual information from the context.
Your factoid question should be formulated in the same style as questions users could ask in a search engine.
This means that your factoid question MUST NOT mention something like "according to the passage" or "context".

Provide your answer as follows:

Output:::
Factoid question: (your factoid question)
Answer: (your answer to the factoid question)

Now here is the context.

Context: {context}\n
Output:::"""

In [19]:
import random

N_GENERATIONS = 10  # We intentionally generate only 10 QA couples here for cost and time considerations

print(f"Generating {N_GENERATIONS} QA couples...")

outputs = []
for sampled_context in tqdm(random.sample(docs_processed, N_GENERATIONS)):
    # Generate QA couple
    output_QA_couple = call_llm(
        llm_client, QA_generation_prompt.format(context=sampled_context.page_content)
    )
    try:
        question = output_QA_couple.split("Factoid question: ")[-1].split("Answer: ")[0]
        answer = output_QA_couple.split("Answer: ")[-1]
        assert len(answer) < 300, "Answer is too long"
        outputs.append(
            {
                "context": sampled_context.page_content,
                "question": question,
                "answer": answer,
                "source_doc": sampled_context.metadata["source"],
            }
        )
    except:
        continue

Generating 10 QA couples...


  0%|          | 0/10 [00:00<?, ?it/s]

In [21]:
display(pd.DataFrame(outputs).head(1))

,context,question,answer,source_doc
0,"```python\nimport tensorflow as tf\n\ntpu = tf.distribute.cluster_resolver.TPUClusterResolver(...)\nstrategy = tf.distribute.TPUStrategy(tpu)\n\nwith strategy.scope():\n tokenizer = AutoTokenizer.from_pretrained(""tf-tpu/unigram-tokenizer-wikitext"")\n config = AutoConfig.from_pretrained(""roberta-base"")\n config.vocab_size = tokenizer.vocab_size\n model = TFAutoModelForMaskedLM.from_config(config) \n```\n\nSimilarly, the optimizer also needs to be initialized under the same strategy scope with which the model is going to be further compiled. Going over the full training code isn’t something we want to do in this post, so we welcome you to read it [here](https://github.com/huggingface/transformers/blob/main/examples/tensorflow/language-modeling-tpu/run_mlm.py). Instead, let’s discuss another key point of — a TensorFlow-native data collator — [`DataCollatorForLanguageModeling`](https://huggingface.co/docs/transformers/main_classes/data_collator#transformers.DataCollatorForLanguageModeling). \n\n`DataCollatorForLanguageModeling` is responsible for masking randomly selected tokens from the input sequence and preparing the labels. By default, we return the results from these collators as NumPy arrays. However, many collators also support returning these values as TensorFlow tensors if we specify `return_tensor=""tf""`. This was crucial for our data pipeline to be compatible with TPU training. \n\nThankfully, TensorFlow provides seamless support for reading files from a GCS bucket:\n\n```python\ntraining_records = tf.io.gfile.glob(os.path.join(args.train_dataset, ""*.tfrecord""))\n```",How do you specify DataCollatorForLanguageModeling to return results as TensorFlow tensors?\n,"return_tensor=""tf"".",huggingface/blog/blob/main/tf_tpu.md


In [22]:
question_groundedness_critique_prompt = """
You will be given a context and a question.
Your task is to provide a 'total rating' scoring how well one can answer the given question unambiguously with the given context.
Give your answer on a scale of 1 to 5, where 1 means that the question is not answerable at all given the context, and 5 means that the question is clearly and unambiguously answerable with the context.

Provide your answer as follows:

Answer:::
Evaluation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here are the question and context.

Question: {question}\n
Context: {context}\n
Answer::: """

question_relevance_critique_prompt = """
You will be given a question.
Your task is to provide a 'total rating' representing how useful this question can be to machine learning developers building NLP applications with the Hugging Face ecosystem.
Give your answer on a scale of 1 to 5, where 1 means that the question is not useful at all, and 5 means that the question is extremely useful.

Provide your answer as follows:

Answer:::
Evaluation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here is the question.

Question: {question}\n
Answer::: """

question_standalone_critique_prompt = """
You will be given a question.
Your task is to provide a 'total rating' representing how context-independent this question is.
Give your answer on a scale of 1 to 5, where 1 means that the question depends on additional information to be understood, and 5 means that the question makes sense by itself.
For instance, if the question refers to a particular setting, like 'in the context' or 'in the document', the rating must be 1.
The questions can contain obscure technical nouns or acronyms like Gradio, Hub, Hugging Face or Space and still be a 5: it must simply be clear to an operator with access to documentation what the question is about.

For instance, "What is the name of the checkpoint from which the ViT model is imported?" should receive a 1, since there is an implicit mention of a context, thus the question is not independent from the context.

Provide your answer as follows:

Answer:::
Evaluation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here is the question.

Question: {question}\n
Answer::: """

In [25]:
print("Generating critique for each QA couple...")
for output in tqdm(outputs):
    evaluations = {
        "groundedness": call_llm(
            llm_client,
            question_groundedness_critique_prompt.format(
                context=output["context"], question=output["question"]
            ),
        ),
        "relevance": call_llm(
            llm_client,
            question_relevance_critique_prompt.format(question=output["question"]),
        ),
        "standalone": call_llm(
            llm_client,
            question_standalone_critique_prompt.format(question=output["question"]),
        ),
    }
    try:
        for criterion, evaluation in evaluations.items():
            score, eval = (
                int(evaluation.split("Total rating: ")[-1].strip()),
                evaluation.split("Total rating: ")[-2].split("Evaluation: ")[1],
            )
            output.update(
                {
                    f"{criterion}_score": score,
                    f"{criterion}_eval": eval,
                }
            )
    except Exception as e:
        continue
    

Generating critique for each QA couple...


  0%|          | 0/10 [00:00<?, ?it/s]

In [26]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

generated_questions = pd.DataFrame.from_dict(outputs)

print("Evaluation dataset before filtering:")
display(
    generated_questions[
        [
            "question",
            "answer",
            "groundedness_score",
            "relevance_score",
            "standalone_score",
        ]
    ]
)
generated_questions = generated_questions.loc[
    (generated_questions["groundedness_score"] >= 4)
    & (generated_questions["relevance_score"] >= 4)
    & (generated_questions["standalone_score"] >= 4)
]
print("============================================")
print("Final evaluation dataset:")
display(
    generated_questions[
        [
            "question",
            "answer",
            "groundedness_score",
            "relevance_score",
            "standalone_score",
        ]
    ]
)

eval_dataset = datasets.Dataset.from_pandas(
    generated_questions, split="train", preserve_index=False
)

Evaluation dataset before filtering:


,question,answer,groundedness_score,relevance_score,standalone_score
0,How do you specify DataCollatorForLanguageModeling to return results as TensorFlow tensors?\n,"return_tensor=""tf"".",5,5,5
1,What type of audio files are used in the Blocks neural instrument coding demo?\n,MP3 and WAV files.,5,4,4
2,How do model files in Transformers aim to be self-contained?\n,"Model files should be as self-contained as possible, so that when you read the code of a specific model, you ideally only have to look into the respective `modeling_....py` file.",5,5,5
3,What is the name of the model used for the 'llama-7b-embeddings' model?\n,llama,5,4,1
4,What type of models can be visualized to reflect stereotypes?\n,Generative models.,5,2,5
5,what is the name of the pre-trained inpainting checkpoint used in the example code?\n,kandinsky-2-2-decoder-inpaint,5,4,5
6,What is the default port used for `sdk` as `docker`?\n,7860,4,5,2
7,What is the current version of the @gradio/upload package in the 0.3.3 release?\n,@gradio/upload@0.3.3,5,2,5
8,What is the name of the model variant used in the example code?\n,seresnet152d,5,3,4
9,What is the package required to install in order to use the cookiecutter command?\n,pip,5,4,5


Final evaluation dataset:


,question,answer,groundedness_score,relevance_score,standalone_score
0,How do you specify DataCollatorForLanguageModeling to return results as TensorFlow tensors?\n,"return_tensor=""tf"".",5,5,5
1,What type of audio files are used in the Blocks neural instrument coding demo?\n,MP3 and WAV files.,5,4,4
2,How do model files in Transformers aim to be self-contained?\n,"Model files should be as self-contained as possible, so that when you read the code of a specific model, you ideally only have to look into the respective `modeling_....py` file.",5,5,5
5,what is the name of the pre-trained inpainting checkpoint used in the example code?\n,kandinsky-2-2-decoder-inpaint,5,4,5
9,What is the package required to install in order to use the cookiecutter command?\n,pip,5,4,5


While it's nice to be able to create our own qa datasets with other llms we can directly get a qa dataset straight from hugging face to save processing power 